# MA Crossover Strategy Research

Full development cycle: hypothesis → backtest → walk-forward → permutation test.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

from src.data.fetcher import YFinanceFetcher
from src.data.store import DataStore
from src.data import df_to_candles
from src.engine.backtest import BacktestEngine, BacktestConfig
from src.analytics.metrics import compute_all_metrics
from src.strategies import STRATEGY_REGISTRY

## Hypothesis

A 10/50 moving average crossover on SPY should capture medium-term trends.
Entry: buy when 10-day SMA crosses above 50-day SMA.
Exit: sell when 10-day SMA crosses below 50-day SMA.

In [ ]:
fetcher = YFinanceFetcher()
store = DataStore(cache_dir="../data/raw")
df = store.fetch_or_cache("SPY", datetime(2015, 1, 1), datetime(2024, 12, 31), fetcher)
candles = df_to_candles(df)

strategy = STRATEGY_REGISTRY["ma_crossover"](fast_period=10, slow_period=50)
config = BacktestConfig(initial_capital=100_000, commission_pct=0.001, slippage_pct=0.0005)

result = BacktestEngine().run(strategy, candles, config, symbol="SPY")
metrics = compute_all_metrics(result)

print("=== Strategy Performance ===")
for k, v in metrics.items():
    if k in ("sharpe_ratio", "sortino_ratio", "cagr", "max_drawdown", "win_rate", "profit_factor", "total_trades"):
        print(f"  {k:<20}: {v:.4f}")

In [ ]:
dates = [pt.date for pt in result.equity_curve]
equity = [pt.equity for pt in result.equity_curve]
drawdown = [pt.drawdown_pct * 100 for pt in result.equity_curve]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
                    subplot_titles=["Portfolio Equity", "Drawdown (%)"])
fig.add_trace(go.Scatter(x=dates, y=equity, name="Equity", line=dict(color="#00b4d8")), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=drawdown, name="Drawdown", fill="tozeroy", line=dict(color="red")), row=2, col=1)
fig.update_layout(height=500, title="MA Crossover Equity Curve (2015-2024)")
fig.show()

## Walk-Forward Analysis

In [ ]:
from src.analytics.walk_forward import WalkForwardAnalyzer

cls = STRATEGY_REGISTRY["ma_crossover"]
analyzer = WalkForwardAnalyzer(
    strategy_cls=cls,
    in_sample_days=252,
    out_of_sample_days=63,
    step_days=63,
    n_optimization_trials=30,
    config=config,
)
wf = analyzer.run(candles)

print(f"Windows: {len(wf.windows)}")
print(f"Combined Sharpe: {wf.combined_sharpe:.4f}")
print(f"Optimization Stability (CV): {wf.optimization_stability:.4f}")

## Permutation Test — Statistical Significance

In [ ]:
from src.analytics.permutation_test import PermutationTester

tester = PermutationTester(
    strategy=strategy,
    candles=candles,
    n_permutations=200,
    metric="sharpe_ratio",
    config=config,
)
perm = tester.run()

print(f"Actual Sharpe:     {perm.actual_metric:.4f}")
print(f"p-value:           {perm.p_value:.4f}")
print(f"Percentile:        {perm.percentile:.1f}%")
print(f"Significant (p<0.05): {perm.is_significant}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=perm.permuted_metrics, nbinsx=40, name="Permuted", marker_color="gray", opacity=0.7))
fig.add_vline(x=perm.actual_metric, line_dash="dash", line_color="#00b4d8",
              annotation_text=f"Actual: {perm.actual_metric:.2f}")
fig.update_layout(title="Permutation Null Distribution vs Actual Strategy",
                  xaxis_title="Sharpe Ratio", yaxis_title="Count", height=400)
fig.show()

## Conclusion

Based on the permutation test, we can assess whether the MA Crossover strategy's
performance is statistically distinguishable from random entry/exit timing.
A p-value < 0.05 suggests the strategy has genuine signal; otherwise performance
may be attributable to chance in this particular time period.